In [14]:
import digitalhub as dh
import pandas as pd
import requests
import os

In [61]:
modes: list[str] = ["CAR"]  ## Choose one or more among: "car_park", "car", "walk", "transit
zoi: str = "allBologna"  ## Choose between "allBologna" and "restrictedAv"
method: str = "extended" ## Choose between "simplified" and "extended"
version: str = "20251003"

In [62]:
if zoi not in ["allBologna", "restrictedAv"]:
    raise ValueError("zoi is not acceptable")
if method not in ["simplified", "extended"]:
    raise ValueError("method is not acceptable")
for mode in modes:
    if mode not in ["CAR_PARK", "TRANSIT", "WALK", "CAR"]:
        raise ValueError("one of modes is not acceptable")

## Prepare OTP
Define function, build, run server

In [30]:
# Project
PROJECT = "test-otp-v2"
project = dh.get_or_create_project(PROJECT)

In [31]:
# OTP server function
func = project.new_function(
    name=f"otp-server-{zoi}",
    kind="container",
    base_image="docker.io/opentripplanner/opentripplanner:latest")

In [32]:
# Go to the Minio console, and create shareable links for each of the artifacts (pbf and gtfs)
GTFS = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0Lzl1OW41bDYxeHVkZmY1YmRtZm1hMXptMC9ndGZzX2dvbW1hZ3Rmc2JvXzIwMjUwNTEzLnppcD9YLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPU0yNzAwMVFIUjBCRzQwQjlHN1BXJTJGMjAyNTEwMDMlMkZ1cy1lYXN0LTElMkZzMyUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjUxMDAzVDEwMDAwOVomWC1BbXotRXhwaXJlcz00MzE5OSZYLUFtei1TZWN1cml0eS1Ub2tlbj1leUpoYkdjaU9pSklVelV4TWlJc0luUjVjQ0k2SWtwWFZDSjkuZXlKaFkyTmxjM05MWlhraU9pSk5NamN3TURGUlNGSXdRa2MwTUVJNVJ6ZFFWeUlzSW1GMFgyaGhjMmdpT2lKWlFWQXhPQzF2UVdaSllWRnVOWHBpWW5GUUxXOUJJaXdpWVhWa0lqb2lXVEk1U1ZkSVFsRldWWGhWWVVSR2JGTjZSbEJPUmxKdlRXdGFSbEl4YUVraUxDSmhkWFJvWDNScGJXVWlPakUzTlRrME56a3pORE1zSW1GMWRHaHZjbWwwYVdWeklqcGJYU3dpWVhwd0lqb2lXVEk1U1ZkSVFsRldWWGhWWVVSR2JGTjZSbEJPUmxKdlRXdGFSbEl4YUVraUxDSmxiV0ZwYkNJNkltRmlkWEo2WVdOamFHbEFabUpyTG1WMUlpd2laVzFoYVd4ZmRtVnlhV1pwWldRaU9uUnlkV1VzSW1WNGNDSTZNVGMxT1RVeU1qazROQ3dpWm1GdGFXeDVYMjVoYldVaU9pSkNkWEo2WVdOamFHa2lMQ0puYVhabGJsOXVZVzFsSWpvaVFYSnBZVzV1WVNJc0ltbGhkQ0k2TVRjMU9UUTNPVGM0Tml3aWFXUWlPaUp2Y0dWdWFXUWlMQ0pwYzNNaU9pSm9kSFJ3Y3pvdkwyRmhZeTVpYjJ4dloyNWhaR2xuYVhSaGJIUjNhVzR1YVhRaUxDSnFkR2tpT2lKak1WUk5hV1pNYUdoZmREbHZVRzk0ZGpkRFJ6RjFjQzEzZEZraUxDSnNiMk5oYkdVaU9pSmxiaUlzSW01aVppSTZNVGMxT1RRM09UYzROaXdpY0dsamRIVnlaU0k2SW1oMGRIQnpPaTh2YkdnekxtZHZiMmRzWlhWelpYSmpiMjUwWlc1MExtTnZiUzloTDBGRFp6aHZZMHBNVDFoc2VFRjNkMWRNYzA1UldWSkNVRVZFYjNnd1MzRXRZMUZOWVdScGEwb3pXVloyWVdoRk5VeGhhVlo1VVhoRFBYTTVOaTFqSWl3aWNHOXNhV041SWpvaWNtVmhaSGR5YVhSbFpHbG5hWFJoYkdoMVlpSXNJbkJ5WldabGNuSmxaRjkxYzJWeWJtRnRaU0k2SW1GaWRYSjZZV05qYUdsQVptSnJMbVYxSWl3aWNtVmhiRzBpT2lKa2FXZHBkR0ZzYUhWaUxXSnZiRzluYm1FaUxDSnliMnhsY3lJNlcxMHNJbk53WVdObFVtOXNaWE1pT2x0ZExDSnpkV0lpT2lKMVgySTFZMll6WldabE1EQmhZelEyTkRjNVlURmhaalpqTWpobFkyRXdNV00zSWl3aWVtOXVaV2x1Wm04aU9pSkhUVlFpZlEuUE5WcmdYaF9nVVFQaVR5SXNXenBGMmtZSV9saVJJbEZ3aHVSaHphWGJDMjZPMFBjVE54NGwtbG1yb2V2RER5bFU5cmYtRklJN2lPdXEzc0hFN2ZXbGcmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0JnZlcnNpb25JZD1udWxsJlgtQW16LVNpZ25hdHVyZT1hYWY1ZGRlZDZkOGYwMzU4NWFmNmJkMzU1Yjg3ZWM0MGZiZTgxMzFkODhlNmI3MjA3MWU4YTg3MDRjZjc0MTdi"
if zoi == "allBologna":
    PBF = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0L3hwd2lkaTVyMXE5ZjAzMG1mbWE0dXJtL2JvbG9nbmEtYXJlYS1maWx0ZXJlZC1wYXJraW5nLXNvcnRlZC5vc20ucGJmP1gtQW16LUFsZ29yaXRobT1BV1M0LUhNQUMtU0hBMjU2JlgtQW16LUNyZWRlbnRpYWw9TTI3MDAxUUhSMEJHNDBCOUc3UFclMkYyMDI1MTAwMyUyRnVzLWVhc3QtMSUyRnMzJTJGYXdzNF9yZXF1ZXN0JlgtQW16LURhdGU9MjAyNTEwMDNUMTAwMTMzWiZYLUFtei1FeHBpcmVzPTQzMTk5JlgtQW16LVNlY3VyaXR5LVRva2VuPWV5SmhiR2NpT2lKSVV6VXhNaUlzSW5SNWNDSTZJa3BYVkNKOS5leUpoWTJObGMzTkxaWGtpT2lKTk1qY3dNREZSU0ZJd1FrYzBNRUk1UnpkUVZ5SXNJbUYwWDJoaGMyZ2lPaUpaUVZBeE9DMXZRV1pKWVZGdU5YcGlZbkZRTFc5Qklpd2lZWFZrSWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKaGRYUm9YM1JwYldVaU9qRTNOVGswTnprek5ETXNJbUYxZEdodmNtbDBhV1Z6SWpwYlhTd2lZWHB3SWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKbGJXRnBiQ0k2SW1GaWRYSjZZV05qYUdsQVptSnJMbVYxSWl3aVpXMWhhV3hmZG1WeWFXWnBaV1FpT25SeWRXVXNJbVY0Y0NJNk1UYzFPVFV5TWprNE5Dd2labUZ0YVd4NVgyNWhiV1VpT2lKQ2RYSjZZV05qYUdraUxDSm5hWFpsYmw5dVlXMWxJam9pUVhKcFlXNXVZU0lzSW1saGRDSTZNVGMxT1RRM09UYzROaXdpYVdRaU9pSnZjR1Z1YVdRaUxDSnBjM01pT2lKb2RIUndjem92TDJGaFl5NWliMnh2WjI1aFpHbG5hWFJoYkhSM2FXNHVhWFFpTENKcWRHa2lPaUpqTVZSTmFXWk1hR2hmZERsdlVHOTRkamREUnpGMWNDMTNkRmtpTENKc2IyTmhiR1VpT2lKbGJpSXNJbTVpWmlJNk1UYzFPVFEzT1RjNE5pd2ljR2xqZEhWeVpTSTZJbWgwZEhCek9pOHZiR2d6TG1kdmIyZHNaWFZ6WlhKamIyNTBaVzUwTG1OdmJTOWhMMEZEWnpodlkwcE1UMWhzZUVGM2QxZE1jMDVSV1ZKQ1VFVkViM2d3UzNFdFkxRk5ZV1JwYTBveldWWjJZV2hGTlV4aGFWWjVVWGhEUFhNNU5pMWpJaXdpY0c5c2FXTjVJam9pY21WaFpIZHlhWFJsWkdsbmFYUmhiR2gxWWlJc0luQnlaV1psY25KbFpGOTFjMlZ5Ym1GdFpTSTZJbUZpZFhKNllXTmphR2xBWm1KckxtVjFJaXdpY21WaGJHMGlPaUprYVdkcGRHRnNhSFZpTFdKdmJHOW5ibUVpTENKeWIyeGxjeUk2VzEwc0luTndZV05sVW05c1pYTWlPbHRkTENKemRXSWlPaUoxWDJJMVkyWXpaV1psTURCaFl6UTJORGM1WVRGaFpqWmpNamhsWTJFd01XTTNJaXdpZW05dVpXbHVabThpT2lKSFRWUWlmUS5QTlZyZ1hoX2dVUVBpVHlJc1d6cEYya1lJX2xpUklsRndodVJoemFYYkMyNk8wUGNUTng0bC1sbXJvZXZERHlsVTlyZi1GSUk3aU91cTNzSEU3ZldsZyZYLUFtei1TaWduZWRIZWFkZXJzPWhvc3QmdmVyc2lvbklkPW51bGwmWC1BbXotU2lnbmF0dXJlPTc1Y2ZjZmRkMzI1ZjEzMDUyNGE5ZTBiMGNkYWFhZjQ2NGExZTIyODg0ZTZmZmI1ZmIyYWNiZTYzNThkYTQ2NWU"
elif zoi == "restrictedAv":
    PBF1 = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0L2hzbnljbnR5aW53ZTJlNWJtZzB2eWM0MS9ib2xvZ25hLWFyZWEtZmlsdGVyZWQtcGFya2luZy1pbnNpZGUtQVYtZm9vdHdheS5vc20ucGJmP1gtQW16LUFsZ29yaXRobT1BV1M0LUhNQUMtU0hBMjU2JlgtQW16LUNyZWRlbnRpYWw9TTI3MDAxUUhSMEJHNDBCOUc3UFclMkYyMDI1MTAwMyUyRnVzLWVhc3QtMSUyRnMzJTJGYXdzNF9yZXF1ZXN0JlgtQW16LURhdGU9MjAyNTEwMDNUMTAwMTA3WiZYLUFtei1FeHBpcmVzPTQzMjAwJlgtQW16LVNlY3VyaXR5LVRva2VuPWV5SmhiR2NpT2lKSVV6VXhNaUlzSW5SNWNDSTZJa3BYVkNKOS5leUpoWTJObGMzTkxaWGtpT2lKTk1qY3dNREZSU0ZJd1FrYzBNRUk1UnpkUVZ5SXNJbUYwWDJoaGMyZ2lPaUpaUVZBeE9DMXZRV1pKWVZGdU5YcGlZbkZRTFc5Qklpd2lZWFZrSWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKaGRYUm9YM1JwYldVaU9qRTNOVGswTnprek5ETXNJbUYxZEdodmNtbDBhV1Z6SWpwYlhTd2lZWHB3SWpvaVdUSTVTVmRJUWxGV1ZYaFZZVVJHYkZONlJsQk9SbEp2VFd0YVJsSXhhRWtpTENKbGJXRnBiQ0k2SW1GaWRYSjZZV05qYUdsQVptSnJMbVYxSWl3aVpXMWhhV3hmZG1WeWFXWnBaV1FpT25SeWRXVXNJbVY0Y0NJNk1UYzFPVFV5TWprNE5Dd2labUZ0YVd4NVgyNWhiV1VpT2lKQ2RYSjZZV05qYUdraUxDSm5hWFpsYmw5dVlXMWxJam9pUVhKcFlXNXVZU0lzSW1saGRDSTZNVGMxT1RRM09UYzROaXdpYVdRaU9pSnZjR1Z1YVdRaUxDSnBjM01pT2lKb2RIUndjem92TDJGaFl5NWliMnh2WjI1aFpHbG5hWFJoYkhSM2FXNHVhWFFpTENKcWRHa2lPaUpqTVZSTmFXWk1hR2hmZERsdlVHOTRkamREUnpGMWNDMTNkRmtpTENKc2IyTmhiR1VpT2lKbGJpSXNJbTVpWmlJNk1UYzFPVFEzT1RjNE5pd2ljR2xqZEhWeVpTSTZJbWgwZEhCek9pOHZiR2d6TG1kdmIyZHNaWFZ6WlhKamIyNTBaVzUwTG1OdmJTOWhMMEZEWnpodlkwcE1UMWhzZUVGM2QxZE1jMDVSV1ZKQ1VFVkViM2d3UzNFdFkxRk5ZV1JwYTBveldWWjJZV2hGTlV4aGFWWjVVWGhEUFhNNU5pMWpJaXdpY0c5c2FXTjVJam9pY21WaFpIZHlhWFJsWkdsbmFYUmhiR2gxWWlJc0luQnlaV1psY25KbFpGOTFjMlZ5Ym1GdFpTSTZJbUZpZFhKNllXTmphR2xBWm1KckxtVjFJaXdpY21WaGJHMGlPaUprYVdkcGRHRnNhSFZpTFdKdmJHOW5ibUVpTENKeWIyeGxjeUk2VzEwc0luTndZV05sVW05c1pYTWlPbHRkTENKemRXSWlPaUoxWDJJMVkyWXpaV1psTURCaFl6UTJORGM1WVRGaFpqWmpNamhsWTJFd01XTTNJaXdpZW05dVpXbHVabThpT2lKSFRWUWlmUS5QTlZyZ1hoX2dVUVBpVHlJc1d6cEYya1lJX2xpUklsRndodVJoemFYYkMyNk8wUGNUTng0bC1sbXJvZXZERHlsVTlyZi1GSUk3aU91cTNzSEU3ZldsZyZYLUFtei1TaWduZWRIZWFkZXJzPWhvc3QmdmVyc2lvbklkPW51bGwmWC1BbXotU2lnbmF0dXJlPTZlYjlkMmRlZTFmYzA3OTkzZjdjODdkZmZiYWQ1NjY1ZThjOGIwOGZhYWFlMzNmM2MyZmNiMTkxNDBmNmU4YTY"
    PBF2 = "http://minio.bolognadigitaltwin.it/api/v1/download-shared-object/aHR0cDovLzEyNy4wLjAuMTo5MDAwL2RhdGFsYWtlL3Rlc3Qtb3RwLXYyL2FydGlmYWN0L2Izcm90MXoyNjFsMWM0ZGJtZm5uejBvdS9ib2xvZ25hLWFyZWEtZmlsdGVyZWQtcGFya2luZy1vdXRzaWRlLUFWLm9zbS5wYmY_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1NMjcwMDFRSFIwQkc0MEI5RzdQVyUyRjIwMjUxMDAzJTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwM1QxMDAwNDdaJlgtQW16LUV4cGlyZXM9NDMyMDAmWC1BbXotU2VjdXJpdHktVG9rZW49ZXlKaGJHY2lPaUpJVXpVeE1pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SmhZMk5sYzNOTFpYa2lPaUpOTWpjd01ERlJTRkl3UWtjME1FSTVSemRRVnlJc0ltRjBYMmhoYzJnaU9pSlpRVkF4T0MxdlFXWkpZVkZ1TlhwaVluRlFMVzlCSWl3aVlYVmtJam9pV1RJNVNWZElRbEZXVlhoVllVUkdiRk42UmxCT1JsSnZUV3RhUmxJeGFFa2lMQ0poZFhSb1gzUnBiV1VpT2pFM05UazBOemt6TkRNc0ltRjFkR2h2Y21sMGFXVnpJanBiWFN3aVlYcHdJam9pV1RJNVNWZElRbEZXVlhoVllVUkdiRk42UmxCT1JsSnZUV3RhUmxJeGFFa2lMQ0psYldGcGJDSTZJbUZpZFhKNllXTmphR2xBWm1KckxtVjFJaXdpWlcxaGFXeGZkbVZ5YVdacFpXUWlPblJ5ZFdVc0ltVjRjQ0k2TVRjMU9UVXlNams0TkN3aVptRnRhV3g1WDI1aGJXVWlPaUpDZFhKNllXTmphR2tpTENKbmFYWmxibDl1WVcxbElqb2lRWEpwWVc1dVlTSXNJbWxoZENJNk1UYzFPVFEzT1RjNE5pd2lhV1FpT2lKdmNHVnVhV1FpTENKcGMzTWlPaUpvZEhSd2N6b3ZMMkZoWXk1aWIyeHZaMjVoWkdsbmFYUmhiSFIzYVc0dWFYUWlMQ0pxZEdraU9pSmpNVlJOYVdaTWFHaGZkRGx2VUc5NGRqZERSekYxY0MxM2RGa2lMQ0pzYjJOaGJHVWlPaUpsYmlJc0ltNWlaaUk2TVRjMU9UUTNPVGM0Tml3aWNHbGpkSFZ5WlNJNkltaDBkSEJ6T2k4dmJHZ3pMbWR2YjJkc1pYVnpaWEpqYjI1MFpXNTBMbU52YlM5aEwwRkRaemh2WTBwTVQxaHNlRUYzZDFkTWMwNVJXVkpDVUVWRWIzZ3dTM0V0WTFGTllXUnBhMG96V1ZaMllXaEZOVXhoYVZaNVVYaERQWE01Tmkxaklpd2ljRzlzYVdONUlqb2ljbVZoWkhkeWFYUmxaR2xuYVhSaGJHaDFZaUlzSW5CeVpXWmxjbkpsWkY5MWMyVnlibUZ0WlNJNkltRmlkWEo2WVdOamFHbEFabUpyTG1WMUlpd2ljbVZoYkcwaU9pSmthV2RwZEdGc2FIVmlMV0p2Ykc5bmJtRWlMQ0p5YjJ4bGN5STZXMTBzSW5Od1lXTmxVbTlzWlhNaU9sdGRMQ0p6ZFdJaU9pSjFYMkkxWTJZelpXWmxNREJoWXpRMk5EYzVZVEZoWmpaak1qaGxZMkV3TVdNM0lpd2llbTl1WldsdVptOGlPaUpIVFZRaWZRLlBOVnJnWGhfZ1VRUGlUeUlzV3pwRjJrWUlfbGlSSWxGd2h1Umh6YVhiQzI2TzBQY1ROeDRsLWxtcm9ldkREeWxVOXJmLUZJSTdpT3VxM3NIRTdmV2xnJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCZ2ZXJzaW9uSWQ9bnVsbCZYLUFtei1TaWduYXR1cmU9YTdkODQyZGNiNTk4OTI2MTk2MDJlMTlmNmEyOGZjNTI3MzBhNWY4YjFiNTkyNmY0OGM0NWE3ODAyODg0ZjRjYw"
# Updated 2025-10-03

In [33]:
# Run the OTP server
if zoi == "allBologna": 
    instructions = [
    "apt-get -y update; apt-get -y install curl",
    "mkdir -p /var/opentripplanner",
    f"curl -L {PBF} -o /var/opentripplanner/osm-tn-bologna.osm.pbf",
    f"curl -L {GTFS} -o /var/opentripplanner/gtfs-pts-bus-bologna.zip"
]
elif zoi == "restrictedAv":
    instructions = [
    "apt-get -y update; apt-get -y install curl",
    "mkdir -p /var/opentripplanner",
    f"curl -L {PBF1} -o /var/opentripplanner/osm-modified-tn-bologna-inside-av.osm.pbf",
    f"curl -L {PBF2} -o /var/opentripplanner/osm-tn-bologna-outside-av.osm.pbf",
    f"curl -L {GTFS} -o /var/opentripplanner/gtfs-pts-bus-bologna.zip"
]

build_run = func.run(action="build", instructions=instructions)
build_status = build_run.wait()
# After running, wait before serving!!!

2025-10-03 10:11:43,137 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:11:48,153 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:11:53,185 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:11:58,220 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:12:03,251 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:12:08,280 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:12:13,306 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:12:18,336 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:12:23,361 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:12:28,394 - INFO - Waiting for run 14078fce922e414590a29ba295eea153 to finish...
2025-10-03 10:12:33,423 - INFO - Run 14078fce922e4

In [34]:
if build_status.status.state == "COMPLETED":
    print(build_status.status.state)
    server_run = func.run(action="serve",
                      args=["--build", "--serve"],
                      service_ports=[{"port": 8080, "target_port": 8080}],
                      service_type='ClusterIP',
                      run_as_user=60000, run_as_group=60000
                    )

COMPLETED


In [35]:
server_status = server_run.wait()
# After running, wait before serving!!!

2025-10-03 10:12:47,108 - INFO - Waiting for run fcca4cb026ed4551957c7838dd048b3a to deploy service.
2025-10-03 10:12:47,137 - INFO - Run fcca4cb026ed4551957c7838dd048b3a service deployed.


## Use OTP

In [53]:
PROJECT = "test-otp-v2"
project = dh.get_or_create_project(PROJECT)

In [54]:
otp_func = project.new_function("otp-processor",
                                    kind="python",
                                    python_version="PYTHON3_10",
                                    code_src="otp_processor.py",
                                    handler="main_platform")

In [63]:
if method == "simplified":
    od_input_key = "store://test-otp-v2/dataitem/table/od-coords-simplified:s471xcn8igf703bbmfo4nn3l"
if method == "extended":
    od_input_key = "store://test-otp-v2/dataitem/table/od-coords-extended:19tqn6505owcb31amfv1qz0g"

url_port = "s-containerserve-aafb747537834bc19790fc5a9479a5e3.digitalhub-bologna:8080"
url_port = "s-containerserve-fcca4cb026ed4551957c7838dd048b3a.digitalhub-bologna:8080"
# Updated 2025-10-03

modes_str =  "_".join(modes)
otp_endpoint = f"http://{url_port}/otp/gtfs/v1"
output_name = f"otp_results_{method}_{zoi}_{modes_str}_v{version}"

In [64]:
otp_run = otp_func.run("job", 
                       inputs={'df': od_input_key},
                       parameters={'main_modes': modes,
                                   'method': method,
                                   'zoi': zoi,
                                   'output_name': output_name,
                                   'otp_endpoint': otp_endpoint})